# LLM Colors Preference Analysis
Analyze and compare results of adaptive preference elicitation (GRUM) from LLMs across models, embeddings, and sampling criteria.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import cosine_similarity

# Set project root to import grums modules if needed
ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", palette="muted")

# Global Configuration
color_names = ["blue", "red", "green", "purple", "yellow"]
color_palette = {"blue": "#1f77b4", "red": "#d62728", "green": "#2ca02c", "purple": "#9467bd", "yellow": "#d6d627"}

## 1. Load Data
We load results from the latest run directory. Each result file contains metadata identifying the model, criterion, and embedding method.

In [2]:
# Set experiment directory
EXP_DIR = ROOT / "results" / "llm" / "llm_colors-20260326-114531"

results_data = []
raw_results = []
output_dir = EXP_DIR / "outputs"

if output_dir.exists():
    for f in sorted(output_dir.glob("*.json")):
        with open(f, "r") as j:
            res = json.load(j)
            
            # Metadata extraction
            criterion = res.get("criterion", "social").capitalize()
            embedding = res.get("embedding_method", "")
            if not embedding:
                embedding = "HS" if "hidden_state" in f.name else "ST"
            else:
                embedding = "HS" if "hidden_state" in embedding else "ST"
            
            model_full = res.get("model_id", "").split("/")[-1]
            model_type = "Instruct" if "Instruct" in model_full else "Pretrained"
            
            # Data extraction
            history = res.get("history", {})
            if not history: continue
            
            last_step = str(sorted(map(int, history.keys()))[-1])
            grum_params = history[last_step]["grum"]
            delta = np.array(grum_params["delta"])
            B = np.array(grum_params["interaction"])
            
            raw_results.append({
                "Model": model_type,
                "Embedding": embedding,
                "Criterion": criterion,
                "delta": delta,
                "B": B
            })
            
            for i, color in enumerate(color_names):
                results_data.append({
                    "Model Type": model_type,
                    "Embedding": embedding,
                    "Criterion": criterion,
                    "Color": color,
                    "Score (Delta)": delta[i]
                })

df = pd.DataFrame(results_data)
print(f"Loaded {len(raw_results)} experiment runs.")

## 1.1 Parameter Consistency and B Matrix Similarity
Comparing the $B$ interaction matrix across alternatives. If the columns of $B$ are highly similar, it implies that agent features drive a **global bias** (affecting all alternatives similarly) rather than **item-specific** personalization.

In [ ]:
similarity_results = []

for r in raw_results:
    B = r['B']
    # Calculate cosine similarity between columns (alternatives)
    # B is (num_features, num_alternatives)
    col_sim = cosine_similarity(B.T)
    # Average off-diagonal elements
    avg_col_sim = (np.sum(col_sim) - B.shape[1]) / (B.shape[1] * (B.shape[1] - 1))
    
    similarity_results.append({
        "Model": r['Model'],
        "Embedding": r['Embedding'],
        "Criterion": r['Criterion'],
        "Avg Col Similarity": avg_col_sim
    })

sim_df = pd.DataFrame(similarity_results)
print("B Matrix Column (Alternative) Similarity:")
display(sim_df.groupby(["Model", "Embedding"])[["Avg Col Similarity"]].mean().round(4))

## 1.2 Detailed Interaction Analysis: Instruct Model
Following the observation that HS interaction patterns appear less distinct than ST, we examine the $B$ matrix for the **Instruct** model across all three elicitation criteria.

In [ ]:
def plot_b_matrix(B, title, ax):
    vmax = np.abs(B).max()
    sns.heatmap(B, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, 
                vmax=vmax, vmin=-vmax,
                xticklabels=color_names, yticklabels=[f"F{i}" for i in range(B.shape[0])])
    ax.set_title(title)
    ax.set_xlabel("Alternative (Color)")
    ax.set_ylabel("Agent Feature Index (PCA)")

instruct_results = [r for r in raw_results if r['Model'] == 'Instruct']
criteria_order = ["Social", "Personalized", "Random"]
embedding_order = ["HS", "ST"]

fig, axes = plt.subplots(3, 2, figsize=(16, 18))

for i, crit in enumerate(criteria_order):
    for j, emb in enumerate(embedding_order):
        ax = axes[i, j]
        matching = [r for r in instruct_results if r['Embedding'] == emb and r['Criterion'] == crit]
        if matching:
            plot_b_matrix(matching[0]['B'], f"Instruct | {emb} | {crit}", ax)
        else:
            ax.set_title(f"Missing: {emb} | {crit}")
            ax.axis('off')

plt.tight_layout()
plt.show()

## 2. Comparative Analysis
Visualizing preferences ($\delta$) across models, embeddings, and criteria.

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
sns.barplot(data=df, x="Color", y="Score (Delta)", hue="Model Type", ax=axes[0], errorbar=None, palette="Set1")
axes[0].set_title("Model Type Comparison")
sns.barplot(data=df, x="Color", y="Score (Delta)", hue="Embedding", ax=axes[1], errorbar=None, palette="Set2")
axes[1].set_title("Embedding Method Comparison")
sns.barplot(data=df, x="Color", y="Score (Delta)", hue="Criterion", ax=axes[2], errorbar=None, palette="Set3")
axes[2].set_title("Criterion Comparison")
for ax in axes: ax.axhline(0, color='black', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Global Perspective
Detailed facet grids showing each combination of Model and Embedding.

In [4]:
g = sns.FacetGrid(df, col="Embedding", row="Model Type", height=4, aspect=1.5, margin_titles=True)
g.map_dataframe(sns.barplot, x="Color", y="Score (Delta)", palette=color_palette, hue="Color", dodge=False, errorbar=None)
g.add_legend(title="Color Item")
plt.subplots_adjust(top=0.9)
g.fig.suptitle("Learned Preferences (Delta) per Model and Embedding")
plt.show()

## 4. The Identifiability Problem: Unbounded Utility Drift
Visualizing historical data without sum-to-zero normalization.

In [ ]:
from grums.experiments.utils import load_experiment_results
artifact_path = "/home/lotanamit/grum4llm/artifacts/llm/colors_unbounded"
unbounded_results = {}
for method in ['hs', 'st']:
    path = os.path.join(artifact_path, method)
    if os.path.exists(path):
        unbounded_results[method] = load_experiment_results(path)

if unbounded_results:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    for i, (method, res) in enumerate(unbounded_results.items()):
        social_runs = [k for k in res.keys() if 'social' in k]
        if not social_runs: continue
        sample_key = social_runs[0]
        history = res[sample_key]['history']
        steps = sorted(map(int, history.keys()))
        deltas = np.array([history[str(s)]['grum']['delta'] for s in steps])
        for j in range(deltas.shape[1]):
            color_name = color_names[j]
            axes[i].plot(steps, deltas[:, j], label=color_name, color=color_palette[color_name])
        axes[i].set_title(f'Unbounded Utility Drift ({method.upper()})')
        axes[i].legend(ncol=2)
    plt.show()